In [33]:
import qiskit
print(qiskit.__version__)

2.5.2


In [34]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (NoiseModel, pauli_error, depolarizing_error, thermal_relaxation_error, ReadoutError)

SEED = 1234   #PRNG, therefore same seed generates the same random numbers

qc = QuantumCircuit(1, 1) # 1 qubit, 1 classical bit
qc.id(0) # identity gate
qc.measure(0,0) # measure qubit 0 into classical bit 0
print(AerSimulator().run(qc, shots=1000, seed_simulator=SEED).result().get_counts())


# 1) bit flip noise x error by 0.05 of probability
bit_flip = pauli_error([("X", 0.05), ("I", 0.95)])
noise = NoiseModel() #making empty noise model
noise.add_all_qubit_quantum_error(bit_flip, ["id"]) #applying bitflip probability to every id gate in QC

print(noise)

noisy = AerSimulator(noise_model=noise)
print(noisy.run(qc, shots=1000, seed_simulator=SEED).result().get_counts())

{'0': 1000}
NoiseModel:
  Basis gates: ['cx', 'id', 'rz', 'sx']
  Instructions with noise: ['id']
  All-qubits errors: ['id']
{'1': 48, '0': 952}


In [54]:
#Custom noise model
#ibm_kingston
import qiskit_aer.noise as noise
x_error_prob = 2.668e-4
sx_error_prob = 2.668E-4
cz_error_prob = 1.961e-3
id_error_prob = 2.668e-4
init_error_prob = 2.000e-4
T1 = 232.01 #us
T2 = 136.04 #us
single_qubit_gate_length = 32 #ns
#two_qubit_gate_length = 
prob_meas0_prep1 = 0.01196
prob_meas1_prep0 = 0.00696
#z_error is implemented as microwave pulses(it saves the phase information of the qubit cos(wt + phi))

noise_model = noise.NoiseModel()

x_error  = noise.depolarizing_error(2*x_error_prob, 1)
sx_error = noise.depolarizing_error(2*sx_error_prob, 1)
id_error = noise.depolarizing_error(2*id_error_prob, 1)
cz_error = noise.depolarizing_error(4/3*cz_error_prob, 2)
init_error = noise.depolarizing_error(init_error_prob, 1)
noise_model.add_all_qubit_quantum_error(x_error, ['x'])
noise_model.add_all_qubit_quantum_error(sx_error, ['sx'])
noise_model.add_all_qubit_quantum_error(cz_error, ['cz'])
noise_model.add_all_qubit_quantum_error(id_error, ['id'])
noise_model.add_all_qubit_quantum_error(init_error, ['init'])

print(noise_model)

NoiseModel:
  Basis gates: ['cx', 'cz', 'id', 'rz', 'sx', 'x']
  Instructions with noise: ['x', 'init', 'id', 'cz', 'sx']
  All-qubits errors: ['x', 'sx', 'cz', 'id', 'init']


In [55]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

ideal = AerSimulator()
noisy = AerSimulator(noise_model=noise_model)

qc = QuantumCircuit(2)
qc.h(0); qc.cx(0, 1); qc.measure_all()
tqc = transpile(qc, basis_gates=['rz', 'sx', 'x', 'cz', 'id']) #translating h, cx into the basis gates of the noise model 


print("ideal", ideal.run(tqc, shots=10000).result().get_counts())
print("noisy", noisy.run(tqc, shots=10000).result().get_counts())



ideal {'00': 5035, '11': 4965}
noisy {'01': 8, '10': 4, '00': 5004, '11': 4984}


In [56]:
from qiskit.quantum_info import average_gate_fidelity
print(1 - average_gate_fidelity(x_error.to_quantumchannel()))
#why is it /2 of original error probability?


0.00026680000000001147


In [48]:
readout_error = ReadoutError([
    [1 - prob_meas1_prep0, prob_meas1_prep0],   # 실제 |0⟩
    [prob_meas0_prep1, 1 - prob_meas0_prep1],   # 실제 |1⟩
])
noise_model.add_all_qubit_readout_error(readout_error)
print(readout_error)

# P(j|i): probability of reading j (0/1) when the true state is i (0/1)

ReadoutError on 1 qubits. Assignment probabilities:
 P(j|0) =  [0.99304 0.00696]
 P(j|1) =  [0.01196 0.98804]


In [ ]:
print(noise_model)


NoiseModel:
  Basis gates: ['cx', 'cz', 'id', 'rz', 'sx', 'x']
  Instructions with noise: ['x', 'init', 'id', 'measure', 'cz', 'sx']
  All-qubits errors: ['x', 'sx', 'cz', 'id', 'init', 'measure']


In [51]:
print("noisy", noisy.run(tqc, shots=10000).result().get_counts())

noisy {'01': 101, '10': 104, '11': 4890, '00': 4905}


In [57]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager
service = QiskitRuntimeService()
backend = service.backend("ibm_kingston")

qc = QuantumCircuit(2)
qc.h(0); qc.cx(0, 1); qc.measure_all()

# 장비에 맞게 변환 (큐빗 배치까지 정해짐)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([isa_qc], shots=10000)
print(job.job_id())                      

result = job.result()
hw_counts = result[0].data.meas.get_counts()   # measure_all()이 만든 레지스터 이름이 'meas'
print("hardware", hw_counts)

qiskit_runtime_service.__init__:WARNING:2026-10-19 01:32:37,314: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-19 01:32:37,315: Using instance: open-instance, plan: open


dbama9pgv2cc73e73ec0
hardware {'11': 4742, '01': 1794, '00': 2429, '10': 1035}
